# Axum Handbook To Build Api

## start 

## intro



##### 3. Line-by-line: your main.rs

Full file for reference:

```rust
use axum::{
    routing::{get, post},
    Router,
    Json,
    extract::Path,
};
use serde::{Deserialize, Serialize};

#[derive(Serialize)]
struct User {
    id: u32,
    name: String,
}

#[derive(Deserialize)]
struct CreateUser {
    name: String,
}

#[tokio::main]
async fn main() {
    let app = Router::new()
        .route("/", get(root))
        .route("/users/{id}", get(get_user))
        .route("/users", post(create_user));

    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000")
        .await
        .unwrap();

    println!("Server running on http://localhost:3000");

    axum::serve(listener, app).await.unwrap();
}

async fn root() -> &'static str {
    "Hello, Axum!"
}

async fn get_user(Path(id): Path<u32>) -> Json<User> {
    Json(User {
        id,
        name: "Mahmoud".to_string(),
    })
}

async fn create_user(Json(payload): Json<CreateUser>) -> Json<User> {
    Json(User {
        id: 1,
        name: payload.name,
    })
}
```

Now, piece by piece:

###### `use axum::{ routing::{get, post}, Router, Json, extract::Path };`
Imports specific items out of the **axum** crate into local scope. Without `use`, you'd have to write the full path every time, like `axum::Router::new()`. `get`/`post` are functions that wrap your handlers and tag them with an HTTP method.

###### `use serde::{Deserialize, Serialize};`
Imports two **traits** from serde. A trait is like an interface — any struct that implements `Serialize` can be turned into JSON; any struct implementing `Deserialize` can be built *from* JSON.

###### `#[derive(Serialize)] struct User { id: u32, name: String }`
`#[derive(...)]` is a macro attribute — instead of you manually writing the serialization logic, the compiler auto-generates it at compile time. This struct is your response shape: an `id` (unsigned 32-bit integer) and a `name` (owned, heap-allocated string).

###### `#[derive(Deserialize)] struct CreateUser { name: String }`
The *input* shape for creating a user. Notice it only has `name`, not `id` — the client shouldn't be allowed to choose the ID, the server assigns it.

###### `#[tokio::main] async fn main() { ... }`
Rust's real `main()` can't be `async` on its own — there's no runtime to poll futures. This macro rewrites your function behind the scenes into a normal `fn main()` that spins up a tokio runtime and blocks on your async code inside it.

###### `let app = Router::new().route("/", get(root))...`
`Router::new()` creates an empty routing table. Each `.route(path, method(handler))` call registers one endpoint and returns `self`, so you can chain them (the **builder pattern**).

###### `.route("/users/{id}", get(get_user))`
The `{id}` segment is a **capture group** (axum 0.8+ syntax). Whatever the client puts there (e.g. `/users/42`) gets extracted and handed to `get_user` via the `Path` extractor.

######  `let listener = tokio::net::TcpListener::bind("0.0.0.0:3000").await.unwrap();`
Opens a TCP socket listening on all network interfaces, port 3000. `.bind()` is async (it returns a `Future`), so it needs `.await` to actually run. `.unwrap()` panics if binding fails (e.g. port already in use) — fine for learning, not for production.

###### `axum::serve(listener, app).await.unwrap();`
This is the actual event loop — it takes your listener and router, and runs forever, accepting connections and dispatching them to the right handler. The program stays "stuck" here until you kill it (Ctrl+C).

###### `async fn root() -> &'static str { "Hello, Axum!" }`
A handler function. Its return type `&'static str` implements axum's `IntoResponse` trait, so axum automatically turns it into an HTTP 200 response with that text as the body.

###### `async fn get_user(Path(id): Path<u32>) -> Json<User>`
This is an **extractor** pattern. `Path<u32>` tells axum "pull the captured URL segment and parse it as a u32." The `Path(id)` on the left is destructuring — unwrapping the value straight into the variable `id`.

###### `Json(User { id, name: "Mahmoud".to_string() })`
Wraps your struct in axum's `Json` type, which serializes it to a JSON body and sets the `Content-Type: application/json` header automatically. `id,` is Rust field-init shorthand for `id: id,`.

###### `async fn create_user(Json(payload): Json<CreateUser>) -> Json<User>`
Here `Json<CreateUser>` is an extractor that reads the incoming request body, parses it as JSON, and deserializes it into a `CreateUser` struct — if the body doesn't match the shape, axum auto-rejects the request with a 400 before your function even runs.

---

##### 4. The mental model (recap)

Crate = compiled unit of Rust code, published and versioned on crates.io. Your `Cargo.toml` declares *which* crates you depend on; `Cargo.lock` pins the *exact* versions resolved. Axum itself is a thin, ergonomic layer over `hyper` (HTTP) and `tower` (middleware), while `tokio` supplies the async engine that makes any of it run concurrently.

> Next natural step given where you are: extractors deep-dive (`Query`, `State`, custom extractors) or wiring Diesel into this same Router via `State` for your product-variant API.

##### unwrap
`.unwrap()` is how you deal with Rust's `Result<T, E>` and `Option<T>` types when you want to say "just give me the value, and if something went wrong, crash the program right here."

###### The core idea

Many operations in Rust don't return the value directly — they return a wrapper that says "this either worked or it didn't":

```rust
enum Result<T, E> {
    Ok(T),
    Err(E),
}
```

`TcpListener::bind(...)` returns a `Result<TcpListener, std::io::Error>` because binding can fail (port already in use, no permission, etc). Rust forces you to *handle* that possibility — it won't silently let you pretend it succeeded.

`.unwrap()` is the lazy/fast way to handle it:

```rust
let listener = TcpListener::bind("0.0.0.0:3000").await.unwrap();
```

- If the `Result` is `Ok(value)` → `.unwrap()` returns `value`, code continues normally.
- If the `Result` is `Err(e)` → `.unwrap()` **panics** — the program crashes immediately and prints the error.

###### Same idea applies to `Option<T>`

```rust
enum Option<T> {
    Some(T),
    None,
}
```

If you `.unwrap()` a `None`, it also panics. E.g. `some_vec.first().unwrap()` panics if the vec is empty.

###### Why it's fine here, but not for production

For a learning project or a quick prototype, `.unwrap()` is totally reasonable — if the port bind fails, you *want* the program to die loudly so you notice.

In a real API, you'd usually handle it more gracefully:

```rust
let listener = match TcpListener::bind("0.0.0.0:3000").await {
    Ok(l) => l,
    Err(e) => {
        eprintln!("Failed to bind: {e}");
        std::process::exit(1);
    }
};
```

Or propagate the error upward with `?` if the function itself returns a `Result`:

```rust
let listener = TcpListener::bind("0.0.0.0:3000").await?;
```

###### Quick comparison

| Method | On `Ok`/`Some` | On `Err`/`None` |
|---|---|---|
| `.unwrap()` | returns value | panics, generic message |
| `.expect("msg")` | returns value | panics, **your custom message** |
| `?` | returns value | propagates the error up to the caller |
| `match` / `if let` | you decide | you decide |

For your Axum/Diesel API specifically, once you get to real request handlers, `?` combined with a custom error type (that implements `IntoResponse`) is the pattern you'll want instead of `.unwrap()` — a panic in a request handler can take down the whole connection instead of just returning a 500.

# other lib and code help with axum

# MY Notes and Study


## Modules

### 1. What a module actually is

A **module** (`mod`) is a namespace inside a single crate. It doesn't compile separately, doesn't get versioned, and isn't published anywhere — it's purely a way to organize your own code into logical groups (like folders, but enforced by the compiler with visibility rules).

Contrast with a **crate**: a crate is the whole compiled package (e.g. `axum`, or your own binary). A crate can contain many modules. Modules never leave the crate they're declared in.

```
crate (axum_rust_api_handbook)
 ├── module: models
 │     └── module: user
 ├── module: handlers
 │     └── module: user
 └── module: routes
       └── module: user
```

---

### 2. Declaring (creating) a module

You create a module with the `mod` keyword. There are three ways to give it a body:

### A. Inline (body written right there)

```rust
mod math {
    pub fn add(a: i32, b: i32) -> i32 {
        a + b
    }
}
```

Good for small, self-contained code. Rarely used for anything beyond quick examples or tests.

### B. Separate file, modern style (no `mod.rs`)

```
src/
├── main.rs
├── models.rs        // declares submodules
└── models/
    └── user.rs
```

```rust
// main.rs
mod models;   // tells Rust: look for models.rs (or models/mod.rs)
```

```rust
// models.rs
pub mod user;   // tells Rust: look for models/user.rs
```

```rust
// models/user.rs
pub struct User {
    pub id: u32,
    pub name: String,
}
```

### C. Separate file, older style (`mod.rs`)

```
src/
├── main.rs
└── models/
    ├── mod.rs        // same role as models.rs above
    └── user.rs
```

```rust
// models/mod.rs
pub mod user;
```

Both B and C compile identically. Style B is what modern tooling (rustfmt, `cargo new`) nudges you toward since Rust 2018, but you'll see `mod.rs` constantly in real-world codebases, including probably your own MERN-adjacent habits translated over. Either is fine — pick one and stay consistent.

> **Rule of thumb:** the file path mirrors the module path. `crate::models::user::User` lives at `src/models/user.rs`.

---

### 3. Visibility: `pub` and why exports don't happen automatically

By default, **everything in Rust is private** to the module it's defined in (and that module's children). Nothing is exported unless you explicitly say so.

```rust
mod models {
    pub mod user;   // must be pub, or nothing outside `models` can see it
}
```

```rust
// models/user.rs
pub struct User {        // pub struct = visible outside this module
    pub id: u32,          // pub field = visible outside this module
    name: String,         // NOT pub = private, only accessible within user.rs
}

pub fn new_user(name: String) -> User {   // pub fn = callable from outside
    User { id: 1, name }
}
```

### Visibility levels

| Modifier | Meaning |
|---|---|
| *(nothing)* | Private — visible only in this module and its children |
| `pub` | Public — visible to anyone who can reach this module |
| `pub(crate)` | Public within the current crate only, hidden from external users of your library |
| `pub(super)` | Visible only to the parent module |
| `pub(in path)` | Visible only within a specific module path you name |

For an application (binary crate) like yours, `pub` vs `pub(crate)` barely matters since nothing outside your own crate ever sees it. `pub(crate)` becomes more relevant if you ever split this into a library crate that other projects depend on.

---

### 4. Importing (`use`)

`mod` declares that a module *exists*. `use` brings items *into scope* so you can refer to them without typing the full path every time.

```rust
// Without `use` — fully qualified path every time
let u = crate::models::user::User { id: 1, name: "Mahmoud".into() };

// With `use` — import once, then use the short name
use crate::models::user::User;
let u = User { id: 1, name: "Mahmoud".into() };
```

### Common `use` patterns

```rust
// Single item
use crate::models::user::User;

// Multiple items from the same module
use crate::models::user::{User, CreateUser};

// Everything public in a module (generally discouraged — unclear where things come from)
use crate::models::user::*;

// Renaming on import, to avoid a name clash
use crate::models::user::User as UserModel;

// External crate imports work the same way
use axum::{routing::get, Router, Json};
use serde::{Deserialize, Serialize};
```

### `crate::`, `self::`, `super::`

These are path prefixes that anchor where Rust starts looking:

```rust
use crate::models::user::User;   // start from the crate root
use self::helpers::format_name;  // start from the current module
use super::User;                 // start from the parent module
```

`crate::` is what you'll use constantly — it means "start from `main.rs` (or `lib.rs`) and walk down."

---

### 5. Exporting (re-exports with `pub use`)

Sometimes you want to flatten a deep path for convenience. That's what `pub use` (a **re-export**) does — it takes something that's already public somewhere, and makes it *also* accessible from a shorter path.

```rust
// models/mod.rs
mod user;                 // private declaration
pub use user::User;       // re-export just the User type at the models:: level
```

Now callers can write:

```rust
use crate::models::User;              // works — re-exported
// instead of:
use crate::models::user::User;        // the "real" deeper path
```

This is extremely common in real crates (including `axum` itself) to give consumers a clean, flat public API while keeping the internal file structure however you like.

---

### 6. Full worked example (matches your API structure)

```
src/
├── main.rs
├── models/
│   ├── mod.rs
│   └── user.rs
├── handlers/
│   ├── mod.rs
│   └── user.rs
└── routes/
    ├── mod.rs
    └── user.rs
```

```rust
// models/user.rs
use serde::{Deserialize, Serialize};

#[derive(Serialize)]
pub struct User {
    pub id: u32,
    pub name: String,
}

#[derive(Deserialize)]
pub struct CreateUser {
    pub name: String,
}
```

```rust
// models/mod.rs
pub mod user;
```

```rust
// handlers/user.rs
use axum::{extract::Path, Json};
use crate::models::user::{User, CreateUser};   // import from models

pub async fn get_user(Path(id): Path<u32>) -> Json<User> {
    Json(User { id, name: "Mahmoud".to_string() })
}

pub async fn create_user(Json(payload): Json<CreateUser>) -> Json<User> {
    Json(User { id: 1, name: payload.name })
}
```

```rust
// handlers/mod.rs
pub mod user;
```

```rust
// routes/user.rs
use axum::{routing::{get, post}, Router};
use crate::handlers::user::{get_user, create_user};   // import from handlers

pub fn user_routes() -> Router {
    Router::new()
        .route("/users/{id}", get(get_user))
        .route("/users", post(create_user))
}
```

```rust
// routes/mod.rs
pub mod user;
```

```rust
// main.rs
mod models;      // declare top-level modules
mod handlers;
mod routes;

use routes::user::user_routes;   // import the one thing main.rs needs

#[tokio::main]
async fn main() {
    let app = user_routes();
    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000").await.unwrap();
    axum::serve(listener, app).await.unwrap();
}
```

---

### 7. Quick reference

| Keyword | Purpose |
|---|---|
| `mod name;` | Declare a module, load it from `name.rs` or `name/mod.rs` |
| `mod name { ... }` | Declare a module inline, body in the same file |
| `pub` | Make an item visible outside its module |
| `use path::Item;` | Bring an item into scope under a short name |
| `pub use path::Item;` | Re-export — make an item accessible from a shorter/different path |
| `crate::` | Path prefix: start from the crate root |
| `super::` | Path prefix: start from the parent module |
| `self::` | Path prefix: start from the current module |

### 8. Common beginner mistakes

- **Forgetting `pub`** — you declare a struct/fn but forget `pub`, then get a "private" compile error when another module tries to use it.
- **Forgetting to declare the module at all** — creating `src/models/user.rs` but never writing `mod models;` in `main.rs`. Rust won't auto-discover files; every module must be explicitly declared somewhere in the tree.
- **Mixing `mod.rs` and non-`mod.rs` style for the same module** — pick one convention and stick to it across the project.
- **Overusing `use ...::*`** — it works, but makes it unclear where a given name came from when reading the code later.

## MVC 
**Model → Controller (Handler) → Router**, and treat the JSON response itself as the "view."

### The mapping

| Classic MVC | Axum equivalent | What it holds |
|---|---|---|
| Model | `models/` | Structs (`User`, `CreateUser`) + DB logic (Diesel queries) |
| View | The `Json<T>` response itself | No template — the struct *is* the view since it just serializes |
| Controller | `handlers/` | Functions like `get_user`, `create_user` — the request/response logic |
| Routes (not in classic MVC, but needed) | `routes/` | Wires paths to handler functions |

### Suggested folder structure

```
src/
├── main.rs
├── models/
│   ├── mod.rs
│   └── user.rs
├── handlers/
│   ├── mod.rs
│   └── user.rs
└── routes/
    ├── mod.rs
    └── user.rs
```

### 1. `models/user.rs` — the "M"

```rust
use serde::{Deserialize, Serialize};

#[derive(Serialize)]
pub struct User {
    pub id: u32,
    pub name: String,
}

#[derive(Deserialize)]
pub struct CreateUser {
    pub name: String,
}
```

### 2. `handlers/user.rs` — the "C"

```rust
use axum::{extract::Path, Json};
use crate::models::user::{User, CreateUser};

pub async fn get_user(Path(id): Path<u32>) -> Json<User> {
    Json(User {
        id,
        name: "Mahmoud".to_string(),
    })
}

pub async fn create_user(Json(payload): Json<CreateUser>) -> Json<User> {
    Json(User {
        id: 1,
        name: payload.name,
    })
}

pub async fn root() -> &'static str {
    "Hello, Axum!"
}
```

### 3. `routes/user.rs`

```rust
use axum::{routing::{get, post}, Router};
use crate::handlers::user::{root, get_user, create_user};

pub fn user_routes() -> Router {
    Router::new()
        .route("/", get(root))
        .route("/users/{id}", get(get_user))
        .route("/users", post(create_user))
}
```

### 4. The `mod.rs` files (just re-export)

```rust
// models/mod.rs
pub mod user;

// handlers/mod.rs
pub mod user;

// routes/mod.rs
pub mod user;
```

### 5. `main.rs` — now just wiring

```rust
mod models;
mod handlers;
mod routes;

use routes::user::user_routes;

#[tokio::main]
async fn main() {
    let app = user_routes();

    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000")
        .await
        .unwrap();

    println!("Server running on http://localhost:3000");

    axum::serve(listener, app).await.unwrap();
}
```

### Why this scales for your Diesel project

When you add Diesel, this split becomes even more valuable:

- **`models/`** grows to hold both the API-facing structs *and* the Diesel `#[derive(Queryable)]` DB row structs, plus your actual query functions (e.g. `User::find_by_id(conn, id)`).
- **`handlers/`** stays thin — it just calls the model function and wraps the result in `Json`, no SQL leaking into the handler.
- **`routes/`** stays a pure map of URL → handler, easy to scan.

As your product-variant API grows (products, variants, stock_movements), you'd just add `models/product.rs`, `handlers/product.rs`, `routes/product.rs` following the same pattern — each domain gets its own trio of files, then you merge routers with `.merge()` or `.nest()` in `main.rs`.

Want me to show you `.nest()` for grouping routes under a prefix (like `/api/v1/users`), since that's the natural next piece once you have multiple resource types?

### Main src after apply MVC


```rs
mod models;
mod handlers;
mod routes;

use axum::Router;
use routes::{user::user_routes, product::product_routes};

#[tokio::main]
async fn main() {
    let app = Router::new()
        .merge(user_routes())
        .merge(product_routes());

    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000")
        .await
        .unwrap();

    println!("Server running on http://localhost:3000");

    axum::serve(listener, app).await.unwrap();
}
```

#### with orm

```rs
mod models;
mod handlers;
mod routes;

use axum::Router;
use diesel::r2d2::{ConnectionManager, Pool};
use diesel::PgConnection;
use routes::{user::user_routes, product::product_routes};

pub type DbPool = Pool<ConnectionManager<PgConnection>>;

#[tokio::main]
async fn main() {
    let manager = ConnectionManager::<PgConnection>::new("postgres://...");
    let pool: DbPool = Pool::builder()
        .build(manager)
        .expect("Failed to create pool");

    let app = Router::new()
        .merge(user_routes())
        .merge(product_routes())
        .with_state(pool);

    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000")
        .await
        .unwrap();

    println!("Server running on http://localhost:3000");

    axum::serve(listener, app).await.unwrap();
}
```



## Add other Routes + how work with modules 